In [1]:
pip install ultralytics opencv-python numpy

Note: you may need to restart the kernel to use updated packages.


In [1]:
import cv2
import numpy as np
import os

# ---------- CONFIG ----------
VIDEO_PATH = "couple1.mp4"          # change if your file name is different
OUTPUT_PATH = "couple1_trimmed.mp4"
PRE_SECONDS = 4.0                # seconds before fall
POST_SECONDS = 1.0               # seconds after fall
# -----------------------------


def find_main_motion_peak(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0 or np.isnan(fps):
        fps = 25.0  # reasonable fallback

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    motion_scores = []
    prev_gray = None
    frame_idx = 0

    print("[INFO] Scanning video for motion peak...")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (21, 21), 0)

        if prev_gray is None:
            prev_gray = gray
            frame_idx += 1
            continue

        diff = cv2.absdiff(prev_gray, gray)
        _, thresh = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)
        thresh = cv2.dilate(thresh, None, iterations=2)

        score = np.sum(thresh)
        motion_scores.append(score)

        prev_gray = gray
        frame_idx += 1

    cap.release()

    if not motion_scores:
        raise RuntimeError("No frames to analyze for motion.")

    motion_scores = np.array(motion_scores, dtype=np.float64)

    # Smooth the motion curve a bit so we don’t pick a single noisy frame
    kernel = np.ones(7) / 7.0
    smoothed = np.convolve(motion_scores, kernel, mode="same")

    peak_idx = int(np.argmax(smoothed))  # index in motion_scores list
    # motion_scores start from frame 1 (we skipped the very first frame)
    peak_frame = peak_idx + 1

    peak_time = peak_frame / fps

    print(f"[INFO] Peak motion at frame {peak_frame} (~{peak_time:.2f} s)")

    return peak_frame, fps, total_frames


def trim_around_peak(video_path, output_path, peak_frame, fps,
                     pre_seconds=1.0, post_seconds=1.0):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    pre_frames = int(pre_seconds * fps)
    post_frames = int(post_seconds * fps)

    start_frame = max(0, peak_frame - pre_frames)
    end_frame = min(total_frames - 1, peak_frame + post_frames)

    print(f"[INFO] Trimming from frame {start_frame} to {end_frame}")

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if start_frame <= frame_idx <= end_frame:
            out.write(frame)

        frame_idx += 1
        if frame_idx > end_frame:
            break

    cap.release()
    out.release()

    print(f"[DONE] Saved trimmed clip to: {output_path}")


def main():
    if not os.path.exists(VIDEO_PATH):
        raise FileNotFoundError(f"Input video not found: {VIDEO_PATH}")

    peak_frame, fps, total_frames = find_main_motion_peak(VIDEO_PATH)
    trim_around_peak(VIDEO_PATH, OUTPUT_PATH, peak_frame, fps,
                     pre_seconds=PRE_SECONDS, post_seconds=POST_SECONDS)


if __name__ == "__main__":
    main()


[INFO] Scanning video for motion peak...
[INFO] Peak motion at frame 271 (~10.84 s)
[INFO] Trimming from frame 171 to 296
[DONE] Saved trimmed clip to: couple1_trimmed.mp4
